# Ders 11 - Model Bağlamı Protokolü (MCP)

**Model Bağlamı Protokolü (MCP)**, ajanların çalışma zamanında araçları, kaynakları ve veri kaynaklarını dinamik olarak keşfetmelerine ve kullanmalarına olanak tanıyan açık bir standarttır. Araçları bir ajana sabit kodlamak yerine, MCP ajanların yetenekleri talep üzerine açığa çıkaran harici sunuculara bağlanmasına olanak tanır.

Bu derste şunları öğreneceksiniz:
- MCP'nin ne olduğunu ve ajan sistemleri için neden önemli olduğunu
- MCP istemci-sunucu mimarisinin nasıl çalıştığını
- MCP tarzı araç keşfini kullanan ajanların nasıl oluşturulacağını


## Kurulum

**Gereksinimler:**
- Dağıtılmış bir modele sahip Azure AI Foundry projesi
- Kimlik doğrulama için `az login` çalıştırın


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) Nedir?

MCP, yapay zeka ajanlarının harici araçları ve veri kaynaklarını keşfetmesi ve bunlarla etkileşim kurması için standart bir yol tanımlar:

- **MCP Server**: Araçları, kaynakları ve komut istemlerini standart bir protokol aracılığıyla erişime açar
- **MCP Client**: Sunuculara bağlanan ve mevcut yetenekleri keşfeden ajan çalışma zamanı
- **Dynamic Discovery**: Ajanların sabit kodlanmış araçlara ihtiyacı yok — çalıştırma zamanında neyin mevcut olduğunu keşfederler

Bu, ajan kodunu değiştirmeden yeni yetenekler eklenebilen genişletilebilir ajan sistemleri oluşturmak için güçlüdür.


## MCP Nasıl Çalışır

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Ajan (MCP istemcisi) bir MCP sunucusuna bağlanır
2. Sunucu kullanılabilir araçların ve şemalarının bir listesini yanıt olarak gönderir
3. Ajan akıl yürütmesi sırasında keşfedilen herhangi bir aracı çağırabilir
4. Sonuçlar aynı protokol aracılığıyla geri iletilir


## MCP Aracı Keşfini Simüle Etme

Gerçek bir MCP sunucusu çalışan bir sunucu süreci gerektirdiğinden, bir MCP'ye bağlı konaklama hizmetinin sağlayacağı şeyleri simüle eden `@tool` işlevlerini kullanarak deseni göstereceğiz.

Üretimde, bu araçlar yerel olarak tanımlanmak yerine bir MCP sunucusundan dinamik olarak keşfedilecektir.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP Tarzı Araçlarla Bir Ajan Oluşturma


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## Üretim Ortamında MCP

Üretim ortamında MCP güçlü kullanım desenleri sağlar:

- **Dinamik araç keşfi**: Ajanlar MCP sunucularına bağlanır ve çalışma zamanında araçları keşfeder
- **Bağımsız mimari**: Araç sağlayıcıları ajanlardan bağımsız olarak güncelleyebilir
- **Kuruluşlar arası paylaşım**: Ekipler, MCP sunucuları aracılığıyla herhangi bir ajanın kullanabileceği yetenekleri paylaşabilir
- **Microsoft Agent Framework desteği**: MAF, `mcp` entegrasyonu aracılığıyla yerleşik MCP istemci desteğine sahiptir

Gerçek bir MCP sunucusunu MAF ile kullanmak için `hosted_mcp_tool()` üzerinden veya MCP istemci entegrasyonu aracılığıyla bağlanırsınız.

**Daha fazla bilgi:**
- [MCP Spesifikasyonu](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Desteği](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Özet

Bu derste şunları öğrendiniz:
- **MCP** ajanlar ile araç sağlayıcıları arasında dinamik araç keşfi için açık bir standarttır
- **İstemci-sunucu mimarisi** ajanların çalışma zamanında yetenekleri keşfetmesini sağlar
- MCP, araçların kod değişikliği olmadan eklenebildiği **genişletilebilir, ayrık ajan sistemlerini** mümkün kılar
- Microsoft Agent Framework üretim kullanımı için **yerleşik MCP desteği** sağlar


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Sorumluluk Reddi:
Bu belge, Yapay Zeka çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba göstermemize rağmen, otomatik çevirilerin hatalar veya yanlışlıklar içerebileceğini lütfen unutmayın. Orijinal belgenin ana dilindeki sürümü yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek herhangi bir yanlış anlaşılma veya yanlış yorumlamadan dolayı sorumluluk kabul etmiyoruz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
